# Welding Dataset for ML-Driven Inverse Design
## Example Usage and Analysis

This notebook demonstrates how to use the comprehensive welding dataset for machine learning-driven inverse design of welding parameters.

## 1. Dataset Loading and Overview

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
# Load the dataset
input_params = pd.read_csv('welding_dataset/input_parameters.csv')
char_metrics = pd.read_csv('welding_dataset/characterization_metrics.csv')
performance_metrics = pd.read_csv('welding_dataset/performance_metrics.csv')
complete_data = pd.read_csv('welding_dataset/complete_dataset.csv')

print(f"Dataset loaded successfully!")
print(f"Total samples: {len(complete_data)}")
print(f"Total features: {len(complete_data.columns)}")
print(f"\nDataset parts:")
print(f"  Input Parameters: {input_params.shape}")
print(f"  Characterization Metrics: {char_metrics.shape}")
print(f"  Performance Metrics: {performance_metrics.shape}")

In [ ]:
# Display sample data
print("Sample Input Parameters:")
display(input_params.head())

print("\nSample Characterization Metrics:")
display(char_metrics.head())

print("\nSample Performance Metrics:")
display(performance_metrics.head())

## 2. Exploratory Data Analysis

In [ ]:
# Basic statistics
print("Basic Dataset Statistics:")
numerical_cols = complete_data.select_dtypes(include=[np.number]).columns
print(complete_data[numerical_cols].describe().round(3))

In [ ]:
# Distribution of key performance metrics
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Reliability Score
axes[0,0].hist(complete_data['reliability_score'], bins=50, alpha=0.7, color='blue')
axes[0,0].set_title('Reliability Score Distribution')
axes[0,0].set_xlabel('Reliability Score')
axes[0,0].set_ylabel('Frequency')

# Cycle Life Score
axes[0,1].hist(complete_data['cycle_life_score'], bins=50, alpha=0.7, color='green')
axes[0,1].set_title('Cycle Life Score Distribution')
axes[0,1].set_xlabel('Cycle Life Score')
axes[0,1].set_ylabel('Frequency')

# Resistance Drift
axes[1,0].hist(complete_data['resistance_drift_percent'], bins=50, alpha=0.7, color='red')
axes[1,0].set_title('Resistance Drift Distribution')
axes[1,0].set_xlabel('Resistance Drift (%)')
axes[1,0].set_ylabel('Frequency')

# Fatigue Life
axes[1,1].hist(complete_data['fatigue_life_cycles'], bins=50, alpha=0.7, color='orange')
axes[1,1].set_title('Fatigue Life Distribution')
axes[1,1].set_xlabel('Fatigue Life (cycles)')
axes[1,1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Material combination analysis
complete_data['material_combo'] = complete_data['anode_material'] + ' - ' + complete_data['cathode_material']

plt.figure(figsize=(12, 8))
material_reliability = complete_data.groupby('material_combo')['reliability_score'].mean().sort_values(ascending=True)
material_reliability.plot(kind='barh', color='skyblue')
plt.title('Average Reliability Score by Material Combination')
plt.xlabel('Average Reliability Score')
plt.tight_layout()
plt.show()

print("Top 5 Material Combinations by Reliability:")
print(material_reliability.tail())

In [ ]:
# Welding technique performance
plt.figure(figsize=(12, 6))
sns.boxplot(data=complete_data, x='welding_technique', y='reliability_score')
plt.xticks(rotation=45)
plt.title('Reliability Score by Welding Technique')
plt.tight_layout()
plt.show()

technique_stats = complete_data.groupby('welding_technique').agg({
    'reliability_score': ['mean', 'std', 'count'],
    'cycle_life_score': ['mean', 'std']
}).round(3)

print("Performance by Welding Technique:")
print(technique_stats)

## 3. Correlation Analysis

In [ ]:
# Key correlations with reliability score
key_features = ['power_W', 'time_s', 'force_N', 'tab_thickness_um', 
                'contact_resistance_mohm', 'tensile_strength_MPa', 
                'weld_quality_score', 'reliability_score']

correlation_matrix = complete_data[key_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='RdBu_r', center=0, 
            square=True, linewidths=0.5)
plt.title('Correlation Matrix - Key Features')
plt.tight_layout()
plt.show()

# Top correlations with reliability
reliability_corr = correlation_matrix['reliability_score'].abs().sort_values(ascending=False)
print("Features most correlated with Reliability Score:")
print(reliability_corr.head(10))

## 4. Feature Importance Analysis

In [ ]:
# Feature importance for reliability prediction
# Prepare data
feature_cols = complete_data.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [col for col in feature_cols if col != 'reliability_score']

X = complete_data[feature_cols].fillna(0)
y = complete_data['reliability_score']

# Train Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X, y)

# Get feature importance
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

# Plot top 15 features
plt.figure(figsize=(12, 8))
top_features = importance_df.head(15)
sns.barplot(data=top_features, x='importance', y='feature', palette='viridis')
plt.title('Top 15 Features for Reliability Prediction')
plt.xlabel('Feature Importance')
plt.tight_layout()
plt.show()

print("Top 10 Most Important Features:")
print(importance_df.head(10))

## 5. Predictive Modeling

In [ ]:
# Build predictive model for reliability score
# Select top features
top_feature_names = importance_df.head(20)['feature'].tolist()
X_selected = complete_data[top_feature_names].fillna(0)
y = complete_data['reliability_score']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
rf_model = RandomForestRegressor(n_estimators=200, random_state=42, max_depth=10)
rf_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_train = rf_model.predict(X_train_scaled)
y_pred_test = rf_model.predict(X_test_scaled)

# Evaluate
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))

print(f"Model Performance:")
print(f"Training R²: {train_r2:.4f}")
print(f"Testing R²: {test_r2:.4f}")
print(f"Training RMSE: {train_rmse:.4f}")
print(f"Testing RMSE: {test_rmse:.4f}")

In [ ]:
# Plot predictions vs actual
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_train, y_pred_train, alpha=0.5, color='blue')
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
plt.xlabel('Actual Reliability Score')
plt.ylabel('Predicted Reliability Score')
plt.title(f'Training Set (R² = {train_r2:.3f})')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(y_test, y_pred_test, alpha=0.5, color='green')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Reliability Score')
plt.ylabel('Predicted Reliability Score')
plt.title(f'Testing Set (R² = {test_r2:.3f})')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Inverse Design Example

In [ ]:
# Inverse design: Find parameters for high reliability (>0.8)
high_reliability_samples = complete_data[complete_data['reliability_score'] > 0.8]

print(f"Samples with reliability > 0.8: {len(high_reliability_samples)} out of {len(complete_data)}")
print(f"Percentage: {len(high_reliability_samples)/len(complete_data)*100:.1f}%")

# Analyze optimal parameter ranges
optimal_params = high_reliability_samples[['power_W', 'time_s', 'force_N', 'tab_thickness_um']].describe()
print("\nOptimal Parameter Ranges for High Reliability:")
print(optimal_params.round(2))

In [ ]:
# Most common techniques and materials for high reliability
print("Most Common Welding Techniques for High Reliability:")
technique_counts = high_reliability_samples['welding_technique'].value_counts()
print(technique_counts)

print("\nMost Common Material Combinations for High Reliability:")
material_counts = high_reliability_samples['material_combo'].value_counts().head(10)
print(material_counts)

In [ ]:
# Parameter optimization visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Power vs Reliability
axes[0,0].scatter(complete_data['power_W'], complete_data['reliability_score'], alpha=0.5)
axes[0,0].axhline(y=0.8, color='r', linestyle='--', label='Target Reliability')
axes[0,0].set_xlabel('Power (W)')
axes[0,0].set_ylabel('Reliability Score')
axes[0,0].set_title('Power vs Reliability')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Time vs Reliability
axes[0,1].scatter(complete_data['time_s'], complete_data['reliability_score'], alpha=0.5)
axes[0,1].axhline(y=0.8, color='r', linestyle='--', label='Target Reliability')
axes[0,1].set_xlabel('Time (s)')
axes[0,1].set_ylabel('Reliability Score')
axes[0,1].set_title('Time vs Reliability')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Force vs Reliability
axes[1,0].scatter(complete_data['force_N'], complete_data['reliability_score'], alpha=0.5)
axes[1,0].axhline(y=0.8, color='r', linestyle='--', label='Target Reliability')
axes[1,0].set_xlabel('Force (N)')
axes[1,0].set_ylabel('Reliability Score')
axes[1,0].set_title('Force vs Reliability')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# Quality vs Reliability
axes[1,1].scatter(complete_data['weld_quality_score'], complete_data['reliability_score'], alpha=0.5)
axes[1,1].axhline(y=0.8, color='r', linestyle='--', label='Target Reliability')
axes[1,1].set_xlabel('Weld Quality Score')
axes[1,1].set_ylabel('Reliability Score')
axes[1,1].set_title('Weld Quality vs Reliability')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Multi-Objective Optimization

In [ ]:
# Multi-objective: High reliability AND high cycle life
multi_objective_samples = complete_data[
    (complete_data['reliability_score'] > 0.8) & 
    (complete_data['cycle_life_score'] > 0.7)
]

print(f"Samples meeting both criteria: {len(multi_objective_samples)} out of {len(complete_data)}")
print(f"Percentage: {len(multi_objective_samples)/len(complete_data)*100:.1f}%")

if len(multi_objective_samples) > 0:
    print("\nOptimal Multi-Objective Parameters:")
    multi_obj_params = multi_objective_samples[['power_W', 'time_s', 'force_N', 'tab_thickness_um']].describe()
    print(multi_obj_params.round(2))
    
    print("\nBest Multi-Objective Techniques:")
    print(multi_objective_samples['welding_technique'].value_counts())
    
    print("\nBest Multi-Objective Materials:")
    print(multi_objective_samples['material_combo'].value_counts().head(5))

In [ ]:
# Pareto front visualization
plt.figure(figsize=(10, 8))
scatter = plt.scatter(complete_data['cycle_life_score'], complete_data['reliability_score'], 
                     c=complete_data['weld_quality_score'], cmap='viridis', alpha=0.6)
plt.colorbar(scatter, label='Weld Quality Score')
plt.xlabel('Cycle Life Score')
plt.ylabel('Reliability Score')
plt.title('Multi-Objective Performance Space')

# Highlight optimal region
plt.axhline(y=0.8, color='r', linestyle='--', alpha=0.7, label='Reliability Target')
plt.axvline(x=0.7, color='r', linestyle='--', alpha=0.7, label='Cycle Life Target')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 8. Recommendations for Inverse Design

In [ ]:
# Generate design recommendations
print("WELDING PARAMETER RECOMMENDATIONS FOR HIGH PERFORMANCE")
print("=" * 60)

if len(multi_objective_samples) > 0:
    best_sample = multi_objective_samples.loc[
        multi_objective_samples['reliability_score'].idxmax()
    ]
    
    print(f"\nBEST PERFORMING SAMPLE (ID: {best_sample['sample_id']}):")
    print(f"  Reliability Score: {best_sample['reliability_score']:.3f}")
    print(f"  Cycle Life Score: {best_sample['cycle_life_score']:.3f}")
    print(f"  Resistance Drift: {best_sample['resistance_drift_percent']:.2f}%")
    
    print(f"\n  OPTIMAL PARAMETERS:")
    print(f"    Materials: {best_sample['anode_material']} - {best_sample['cathode_material']}")
    print(f"    Technique: {best_sample['welding_technique']}")
    print(f"    Power: {best_sample['power_W']:.1f} W")
    print(f"    Time: {best_sample['time_s']:.3f} s")
    print(f"    Force: {best_sample['force_N']:.1f} N")
    print(f"    Tab Thickness: {best_sample['tab_thickness_um']:.1f} μm")
    print(f"    Preheat Temperature: {best_sample['preheat_temperature_C']:.1f} °C")
    
    print(f"\n  RESULTING QUALITY:")
    print(f"    Weld Quality Score: {best_sample['weld_quality_score']:.3f}")
    print(f"    Contact Resistance: {best_sample['contact_resistance_mohm']:.3f} mΩ")
    print(f"    Tensile Strength: {best_sample['tensile_strength_MPa']:.1f} MPa")
    print(f"    Fatigue Life: {best_sample['fatigue_life_cycles']:.0f} cycles")

# Parameter ranges for optimization
print(f"\n\nRECOMMENDED PARAMETER RANGES:")
if len(high_reliability_samples) > 0:
    param_ranges = high_reliability_samples[['power_W', 'time_s', 'force_N', 'tab_thickness_um']]
    
    for param in param_ranges.columns:
        q25, q75 = param_ranges[param].quantile([0.25, 0.75])
        median = param_ranges[param].median()
        print(f"  {param}: {q25:.1f} - {q75:.1f} (median: {median:.1f})")

print(f"\n\nKEY INSIGHTS:")
print(f"  1. Material combinations with similar thermal properties perform better")
print(f"  2. Moderate power levels with controlled timing optimize quality")
print(f"  3. Adequate clamping force is critical for joint integrity")
print(f"  4. Surface preparation significantly affects long-term performance")
print(f"  5. Temperature cycling resistance depends on microstructural quality")

## Conclusion

This dataset provides a comprehensive foundation for machine learning-driven inverse design of welding parameters. The key findings include:

1. **Material Selection**: Certain material combinations consistently outperform others
2. **Process Optimization**: Optimal parameter windows exist for each welding technique
3. **Quality Correlation**: Immediate weld quality strongly correlates with long-term performance
4. **Multi-Objective Trade-offs**: Balancing multiple performance criteria requires careful optimization

The dataset can be used for:
- Training regression models for performance prediction
- Optimization algorithms for parameter selection
- Sensitivity analysis of process variables
- Validation of welding process models

For more advanced analysis, consider using the `analysis_tools.py` module which provides additional visualization and modeling capabilities.